# ESMFold v1 trunk embeddings

Attach the VS Code Colab notebook to an L4 GPU and choose **Run All** once. This sequence-only workflow reads a small manifest from Google Drive and saves one validated `[L,1024]` folding-trunk embedding per protein. It does not use MSAs or save structures.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/dynamic_protein_router/esmfold_io')
INPUT_MANIFEST = DRIVE_ROOT / 'inputs/all_missing_embeddings_manifest.csv'
OUTPUT_DIR = DRIVE_ROOT / 'outputs/all_missing_embeddings'
MODEL_ID = 'facebook/esmfold_v1'
MODEL_REVISION = '75a3841ee059df2bf4d56688166c8fb459ddd97a'
EXTRACTION_CONFIG = {
    'representation': 'folding_trunk_s_s',
    'num_recycles': 0,
    'chunk_size': 128,
    'max_sequence_length': 1022,
    'nonstandard_residue_policy': 'U_to_X',
    'output_dtype': 'float32',
}
MAX_PROTEINS = None

if not INPUT_MANIFEST.is_file():
    raise FileNotFoundError(f'Upload the prepared manifest to {INPUT_MANIFEST}')
gpu = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True
).strip()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('GPU:', gpu)
print('Input:', INPUT_MANIFEST)
print('Output:', OUTPUT_DIR)


In [ ]:
import sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'transformers==4.48.3', 'accelerate>=1.2,<2', 'sentencepiece>=0.2'],
    check=True,
)

import torch
from transformers import AutoTokenizer, EsmForProteinFolding

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = EsmForProteinFolding.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, low_cpu_mem_usage=True
)
model.esm = model.esm.half()
model = model.cuda().eval()
model.trunk.set_chunk_size(EXTRACTION_CONFIG['chunk_size'])
print('Pinned ESMFold v1 is loaded on', next(model.parameters()).device)


In [ ]:
import csv
import json
import os
import time
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np

CENTRAL = ZoneInfo('America/Chicago')

def now():
    return datetime.now(CENTRAL).strftime('%Y-%m-%d %I:%M:%S %p %Z')

def as_bool(value):
    return value is True or str(value).lower() == 'true'

def atomic_json(path, value):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    os.replace(temporary, path)

def write_csv(path, records):
    temporary = path.with_suffix(path.suffix + '.tmp')
    if not records:
        path.unlink(missing_ok=True)
        return
    with temporary.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
    os.replace(temporary, path)

def valid_result(path, row):
    try:
        with np.load(path, allow_pickle=False) as archive:
            if set(archive.files) != {'single', 'metadata'}:
                return False
            single = archive['single']
            metadata = json.loads(str(archive['metadata'].item()))
        return (
            single.dtype == np.float32
            and single.shape == (int(row['sequence_length']), 1024)
            and np.isfinite(single).all()
            and metadata.get('protein_id') == row['protein_id']
            and metadata.get('sequence_sha256') == row['sequence_sha256']
            and metadata.get('model_sequence_sha256') == row['model_sequence_sha256']
            and metadata.get('model_id') == MODEL_ID
            and metadata.get('model_revision') == MODEL_REVISION
            and metadata.get('extraction_config') == EXTRACTION_CONFIG
        )
    except Exception:
        return False

with INPUT_MANIFEST.open(newline='') as handle:
    rows = list(csv.DictReader(handle))
required = {
    'protein_id', 'sequence', 'sequence_sha256', 'model_sequence',
    'model_sequence_sha256', 'sequence_length', 'eligible', 'exclusion_reason'
}
if not rows or required - set(rows[0]):
    raise ValueError('manifest is empty or missing required ESMFold columns')
for row in rows:
    row['sequence_length'] = str(int(float(row['sequence_length'])))
if len({row['sequence_sha256'] for row in rows}) != len(rows):
    raise ValueError('manifest contains duplicate sequence hashes')
eligible = [row for row in rows if as_bool(row['eligible'])]
excluded = [row for row in rows if not as_bool(row['eligible'])]
if MAX_PROTEINS is not None:
    eligible = eligible[:MAX_PROTEINS]
write_csv(OUTPUT_DIR / 'excluded.csv', excluded)
print(f'Validated {len(rows)} rows: {len(eligible)} eligible, {len(excluded)} excluded.')


In [ ]:
progress_path = OUTPUT_DIR / 'progress.json'
started_at = now()
records = []
attempt_seconds = []
for index, row in enumerate(eligible, start=1):
    result_path = OUTPUT_DIR / f"{row['sequence_sha256']}.npz"
    status = 'skipped_existing' if valid_result(result_path, row) else 'running'
    elapsed = 0.0
    error_text = ''
    atomic_json(progress_path, {
        'phase': 'running',
        'timezone': 'Central Time (America/Chicago)',
        'started_at': started_at,
        'updated_at': now(),
        'total_dataset_rows': len(rows),
        'eligible': len(eligible),
        'excluded': len(excluded),
        'processed': index - 1,
        'successful': sum(r['status'] in {'success', 'skipped_existing'} for r in records),
        'failed': sum(r['status'] == 'failed' for r in records),
        'remaining': len(eligible) - index + 1,
        'average_seconds_per_attempt': round(sum(attempt_seconds) / len(attempt_seconds), 2) if attempt_seconds else None,
        'estimated_seconds_remaining': round((sum(attempt_seconds) / len(attempt_seconds)) * (len(eligible) - index + 1)) if attempt_seconds else None,
        'current_protein': row['protein_id'],
    })
    if status == 'running':
        started = time.monotonic()
        try:
            inputs = tokenizer(
                row['model_sequence'], return_tensors='pt', add_special_tokens=False
            )
            if int(inputs['attention_mask'].sum()) != int(row['sequence_length']):
                raise ValueError('tokenizer changed sequence length')
            inputs = {key: value.cuda() for key, value in inputs.items()}
            with torch.inference_mode():
                output = model(**inputs, num_recycles=EXTRACTION_CONFIG['num_recycles'])
            single = output.s_s[0].detach().float().cpu().numpy()
            expected = (int(row['sequence_length']), 1024)
            if single.shape != expected or not np.isfinite(single).all():
                raise ValueError(f'invalid trunk embedding: {single.shape}, expected {expected}')
            metadata = {
                'protein_id': row['protein_id'],
                'sequence_sha256': row['sequence_sha256'],
                'model_sequence_sha256': row['model_sequence_sha256'],
                'model_id': MODEL_ID,
                'model_revision': MODEL_REVISION,
                'extraction_config': EXTRACTION_CONFIG,
                'shape': list(single.shape),
                'dtype': str(single.dtype),
                'created_at': now(),
            }
            temporary = OUTPUT_DIR / f"{row['sequence_sha256']}.part.npz"
            np.savez_compressed(
                temporary, single=single, metadata=np.array(json.dumps(metadata))
            )
            os.replace(temporary, result_path)
            if not valid_result(result_path, row):
                raise RuntimeError('saved checkpoint failed immediate validation')
            status = 'success'
        except Exception as error:
            status = 'failed'
            error_text = f'{type(error).__name__}: {error}'[-2000:]
        finally:
            elapsed = time.monotonic() - started
            attempt_seconds.append(elapsed)
            torch.cuda.empty_cache()
    records.append({
        'protein_id': row['protein_id'],
        'sequence_sha256': row['sequence_sha256'],
        'status': status,
        'seconds': round(elapsed, 2),
        'error': error_text,
    })
    if index % 10 == 0 or status == 'failed' or index == len(eligible):
        write_csv(OUTPUT_DIR / 'status.csv', records)
        write_csv(OUTPUT_DIR / 'failures.csv', [r for r in records if r['status'] == 'failed'])
    average = sum(attempt_seconds) / len(attempt_seconds) if attempt_seconds else None
    remaining = len(eligible) - index
    failures = sum(r['status'] == 'failed' for r in records)
    atomic_json(progress_path, {
        'phase': ('complete_with_failures' if failures else 'complete') if remaining == 0 else 'running',
        'timezone': 'Central Time (America/Chicago)',
        'started_at': started_at,
        'updated_at': now(),
        'total_dataset_rows': len(rows),
        'eligible': len(eligible),
        'excluded': len(excluded),
        'processed': index,
        'successful': sum(r['status'] in {'success', 'skipped_existing'} for r in records),
        'failed': failures,
        'remaining': remaining,
        'average_seconds_per_attempt': round(average, 2) if average is not None else None,
        'estimated_seconds_remaining': round(average * remaining) if average is not None else None,
        'last_completed_protein': row['protein_id'],
    })
    print(f"[{index}/{len(eligible)}] {row['protein_id']}: {status} ({elapsed:.1f}s)", flush=True)

print('Finished. Results:', OUTPUT_DIR)
print('The Colab runtime will be released in 15 seconds.')
time.sleep(15)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as error:
    print('Automatic runtime release failed; disconnect it manually:', error)
